<a href="https://colab.research.google.com/github/Momina3/DATA-SCIENCE-MENTORSHIP---WEEK2/blob/main/Week2_DS_Session.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Week 2 session by Zee Outsourcing :
Messy Data, Cleaning & Model Training

Mentors : Momina Aslam & Faiqa Iman


In [ ]:
## 1. Business Problem

In [ ]:
## 2. Import Dataset
import pandas as pd
df=pd.read_csv('messy_telco.csv')

In [ ]:
df = df.drop_duplicates()


In [ ]:
df["Contract"] = (
    df["Contract"]
    .str.strip()
    .str.lower()
)

In [ ]:
import numpy as np
df.loc[df["tenure"] > 72, "tenure"] = np.nan

In [ ]:
## 3. Data Quality Report

In [ ]:
## 4. Identify Data Problems

In [ ]:
## 5. Cleaning

In [ ]:
### 5.1 Duplicates
### 5.2 Numeric Conversion
### 5.3 Categorical Standardization
### 5.4 Impossible Values
### 5.5 Missing Values

In [ ]:
## 6. AI Code Review

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
## 7. Define X and y
X = df.drop(columns=["Churn", "customerID"])

y = (df["Churn"] == "Yes").astype(int)

In [ ]:
y.value_counts(normalize=True) #show charner and non churner

,proportion
Churn,
0,0.73463
1,0.26537


In [ ]:
num = X.select_dtypes(include="number").columns
cat=X.select_dtypes(include="object").columns
print("numerical:",list(num))
print("categorical:",list(cat))

numerical: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [ ]:
from sklearn.model_selection import StratifiedKFold,cross_validate
cv=StratifiedKFold( #cuts in equal proportion
     n_splits=5,shuffle=True,random_state=42)

In [ ]:
## 8. Evaluation Metric
def evaluate(model):
  scores=cross_validate(
      model,#model you want to train
      X,#features
      y,#target
      cv=cv,
      scoring=["roc_auc","recall"]
  )
  return {
      "roc_auc":scores["test_roc_auc"].mean(),
      "recall":scores["test_recall"].mean()
  }

In [ ]:
## 9. Baseline Model
from sklearn.dummy import DummyClassifier
baseline=DummyClassifier(strategy="prior")

In [ ]:
evaluate(baseline)

{'roc_auc': np.float64(0.5), 'recall': np.float64(0.0)}

In [ ]:
## 10. Preprocessing Pipeline
#pipeline->preprocessing,model training
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])


In [ ]:
from sklearn.preprocessing import OneHotEncoder
categorical_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
from sklearn.compose import ColumnTransformer
preprocessor=ColumnTransformer([
    ("numeric",numeric_pipeline,num),
    ("categorical",categorical_pipeline,cat)
])

In [ ]:
## 11. Logistic Regression
from sklearn.linear_model import LogisticRegression
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [ ]:
evaluate(logreg_pipeline)

{'roc_auc': np.float64(0.8451691701246608),
 'recall': np.float64(0.553746899686026)}

In [ ]:
## 12. Random Forest
from sklearn.ensemble import RandomForestClassifier
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier())
])

In [ ]:
evaluate(rf_pipeline)

{'roc_auc': np.float64(0.8195007447208468),
 'recall': np.float64(0.4863285114191912)}

In [ ]:
## 13. Cross-Validation Results

In [ ]:
## 14. Model Comparison
results=pd.DataFrame({
    "baseline":evaluate(baseline),
    "log_reg":evaluate(logreg_pipeline),
    "rf":evaluate(rf_pipeline)
})

In [ ]:
results

,baseline,log_reg,rf
roc_auc,0.5,0.845169,0.820312
recall,0.0,0.553747,0.471355


In [ ]:
## 15. Conclusions